# Topology Optimization Notebook: Baseline vs Direct 64x64

This notebook compares two pipelines from `gen_rl/`:

- `multistage`: the previous baseline `16 -> 32 -> 64`
- `direct64_exact`: author-aligned exact search directly in `64x64` without progressive refinement

The notebook is intentionally lightweight and delegates all heavy logic to Python modules.

In [ ]:
from __future__ import annotations

import json
import sys
from importlib.util import find_spec
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "gigala").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root containing the gigala package.")

from gigala.topology.topology_optimiz.gen_rl import (
    ProblemConfig,
    make_direct64_refine_env,
    make_refine_env,
    run_search,
)

HAS_GYM = find_spec("gymnasium") is not None
HAS_SB3 = find_spec("stable_baselines3") is not None
HAS_SB3_CONTRIB = find_spec("sb3_contrib") is not None
HAS_MASKABLE_RL = HAS_GYM and HAS_SB3 and HAS_SB3_CONTRIB

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "has_gymnasium": HAS_GYM,
    "has_stable_baselines3": HAS_SB3,
    "has_sb3_contrib": HAS_SB3_CONTRIB,
    "has_maskable_rl": HAS_MASKABLE_RL,
}, indent=2))

In [ ]:
baseline_config = ProblemConfig(
    pipeline_mode="multistage",
    resolution=64,
    volume_target=0.55,
    solver_backend="scipy",
    runtime_budget_hours=0.1,
    enable_rl=False,
    coarse_population=24,
    coarse_generations=12,
    coarse_elite_count=6,
    stage32_top_k=4,
    stage64_top_k=2,
    local_search_steps32=8,
    local_search_steps64=12,
    max_full_evals=300,
    max_rl_full_evals=128,
)

direct_config = ProblemConfig(
    pipeline_mode="direct64_exact",
    resolution=64,
    volume_target=0.55,
    solver_backend="scipy",
    runtime_budget_hours=0.1,
    enable_rl=False,
    direct_population=12,
    direct_elite_count=4,
    direct_offspring_batch=6,
    direct_archive_size=8,
    direct_restart_stagnation_evals=48,
    max_full_evals=300,
    max_rl_full_evals=128,
    workers="auto",
)

baseline_config, direct_config

In [ ]:
baseline_artifacts = run_search(baseline_config)
direct_artifacts = run_search(direct_config)

comparison = {
    "baseline_multistage": {
        "runtime_sec": round(baseline_artifacts.runtime, 2),
        "fea_counts": baseline_artifacts.fea_counts,
        "warnings": baseline_artifacts.warnings,
        "metrics": baseline_artifacts.metrics,
    },
    "direct64_exact": {
        "runtime_sec": round(direct_artifacts.runtime, 2),
        "fea_counts": direct_artifacts.fea_counts,
        "warnings": direct_artifacts.warnings,
        "metrics": direct_artifacts.metrics,
    },
}
print(json.dumps(comparison, indent=2))

In [ ]:
def show_mask_pair(left_title, left_mask, right_title, right_mask):
    yellow_material = ListedColormap(["#ffffff", "#ffd84d"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=120)
    axes[0].imshow(left_mask, cmap=yellow_material, vmin=0, vmax=1)
    axes[0].set_title(left_title)
    axes[0].set_xlabel("Width")
    axes[0].set_ylabel("Height")
    axes[1].imshow(right_mask, cmap=yellow_material, vmin=0, vmax=1)
    axes[1].set_title(right_title)
    axes[1].set_xlabel("Width")
    axes[1].set_ylabel("Height")
    plt.tight_layout()
    plt.show()


show_mask_pair(
    "multistage refined64",
    baseline_artifacts.refined64,
    "direct64 exact best64",
    direct_artifacts.best64,
)

In [ ]:
if baseline_artifacts.refined32 is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=120)
    yellow_material = ListedColormap(["#ffffff", "#ffd84d"])
    axes[0].imshow(baseline_artifacts.coarse16, cmap=yellow_material, vmin=0, vmax=1)
    axes[0].set_title("baseline coarse16")
    axes[1].imshow(baseline_artifacts.refined32, cmap=yellow_material, vmin=0, vmax=1)
    axes[1].set_title("baseline refined32")
    axes[2].imshow(baseline_artifacts.refined64, cmap=yellow_material, vmin=0, vmax=1)
    axes[2].set_title("baseline refined64")
    for axis in axes:
        axis.set_xlabel("Width")
        axis.set_ylabel("Height")
    plt.tight_layout()
    plt.show()
else:
    print("baseline refined32 is unavailable for this config")

## Optional RL Environment Inspection

Both RL environments are optional. The direct environment starts from a native `64x64` seed and uses exact `full64` rewards.

In [ ]:
if HAS_MASKABLE_RL and baseline_artifacts.refined64 is not None:
    baseline_env = make_refine_env(baseline_artifacts.refined64, baseline_config)
    baseline_obs, _ = baseline_env.reset()
    print("baseline env observation shape:", baseline_obs.shape)
    print("baseline valid actions:", int(baseline_env.action_masks().sum()), "out of", baseline_env.action_space.n)
else:
    print("Maskable RL stack is not available; skipping baseline env inspection.")

In [ ]:
if HAS_MASKABLE_RL:
    direct_env = make_direct64_refine_env(direct_artifacts.best64, direct_config)
    direct_obs, direct_info = direct_env.reset()
    print("direct env observation shape:", direct_obs.shape)
    print("direct valid actions:", int(direct_env.action_masks().sum()), "out of", direct_env.action_space.n)
    print("direct initial exact score:", direct_info["evaluation"].score)
else:
    print("Maskable RL stack is not available; skipping direct env inspection.")